# Klingon Translator - Fine-Tuning on Google Colab

Fine-tunes NLLB-200 for English to Klingon translation.

**Runtime:** Go to Runtime > Change runtime type > select **T4 GPU**

## 1. Setup

In [ ]:
!pip install -q transformers sentencepiece datasets sacrebleu accelerate protobuf torch

from google.colab import drive
drive.mount("/content/drive")

import torch
print("GPU available:", torch.cuda.is_available())

## 2. Load Data

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Klingon Translator/data/processed")

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(DATA_DIR / "train.jsonl")
val_data = load_jsonl(DATA_DIR / "val.jsonl")
test_data = load_jsonl(DATA_DIR / "test.jsonl")
print(len(train_data), "train", len(val_data), "val", len(test_data), "test")

## 3. Extend Tokenizer with Klingon

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sentencepiece as spm

BASE_MODEL = "facebook/nllb-200-distilled-600M"
KLINGON_CODE = "tlh_Latn"
ENGLISH_CODE = "eng_Latn"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
print("Base vocab:", len(tokenizer))

In [ ]:
# Train SentencePiece on Klingon text
klingon_lines = [p["tlh"] for p in train_data + val_data]
with open("/content/klingon_corpus.txt", "w") as f:
    f.write(chr(10).join(klingon_lines))

spm.SentencePieceTrainer.train(
    input="/content/klingon_corpus.txt",
    model_prefix="/content/klingon_spm",
    vocab_size=1000, character_coverage=1.0, model_type="bpe")

klingon_sp = spm.SentencePieceProcessor()
klingon_sp.load("/content/klingon_spm.model")

existing = set(tokenizer.get_vocab().keys())
new_tokens = [klingon_sp.id_to_piece(i) for i in range(klingon_sp.get_piece_size())
              if klingon_sp.id_to_piece(i) not in existing]

tokenizer.add_special_tokens({"additional_special_tokens": [KLINGON_CODE]})
tokenizer.add_tokens(new_tokens)
old_size = model.get_input_embeddings().weight.shape[0]
model.resize_token_embeddings(len(tokenizer))

# Init new embeddings from English
eng_id = tokenizer.convert_tokens_to_ids(ENGLISH_CODE)
tlh_id = tokenizer.convert_tokens_to_ids(KLINGON_CODE)
with torch.no_grad():
    model.get_input_embeddings().weight[tlh_id] = model.get_input_embeddings().weight[eng_id].clone()
    mean_emb = model.get_input_embeddings().weight[:old_size].mean(dim=0)
    for i in range(old_size, len(tokenizer)):
        if i != tlh_id:
            model.get_input_embeddings().weight[i] = mean_emb.clone()
print("Extended vocab:", len(tokenizer))

## 4. Bidirectional Training Dataset

In [ ]:
from torch.utils.data import Dataset

class BilingualDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=128):
        self.examples = []
        for p in pairs:
            self.examples.append({"src": p["en"], "tgt": p["tlh"], "src_lang": ENGLISH_CODE, "tgt_lang": KLINGON_CODE})
            self.examples.append({"src": p["tlh"], "tgt": p["en"], "src_lang": KLINGON_CODE, "tgt_lang": ENGLISH_CODE})
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        self.tokenizer.src_lang = ex["src_lang"]
        inputs = self.tokenizer(ex["src"], truncation=True, max_length=self.max_length)
        with self.tokenizer.as_target_tokenizer():
            labels = self.tokenizer(ex["tgt"], truncation=True, max_length=self.max_length)
        inputs["labels"] = [self.tokenizer.convert_tokens_to_ids(ex["tgt_lang"])] + labels["input_ids"]
        return inputs

train_dataset = BilingualDataset(train_data, tokenizer)
val_dataset = BilingualDataset(val_data, tokenizer)
print("Training examples:", len(train_dataset))

## 5. Fine-Tune

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

CKPT = "/content/drive/MyDrive/Klingon Translator/models/checkpoints"
FINAL = "/content/drive/MyDrive/Klingon Translator/models/fine-tuned"

args = Seq2SeqTrainingArguments(
    output_dir=CKPT, num_train_epochs=15,
    per_device_train_batch_size=8, per_device_eval_batch_size=8,
    gradient_accumulation_steps=4, learning_rate=2e-5,
    weight_decay=0.01, warmup_steps=100, fp16=True,
    eval_strategy="epoch", save_strategy="epoch",
    save_total_limit=3, load_best_model_at_end=True,
    predict_with_generate=True, logging_steps=50, report_to="none")

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_dataset,
    eval_dataset=val_dataset, data_collator=collator, tokenizer=tokenizer)

trainer.train()

## 6. Save & Test

In [ ]:
model.save_pretrained(FINAL)
tokenizer.save_pretrained(FINAL)
print("Saved to:", FINAL)

model.eval()
for sent in ["Today is a good day to die.", "Success!", "What do you want?"]:
    tokenizer.src_lang = ENGLISH_CODE
    inp = tokenizer(sent, return_tensors="pt").to(model.device)
    tid = tokenizer.convert_tokens_to_ids(KLINGON_CODE)
    with torch.no_grad():
        out = model.generate(**inp, forced_bos_token_id=tid, max_length=128)
    print(sent, "->", tokenizer.decode(out[0], skip_special_tokens=True))